# Вводная часть

## Проблематика
Автоматизаторы тестирования сталкиваются с проблемой нестабльности скриншотов для регрессионной проверки верстки UI. Это и динамические элементы, неожиданные баннеры, сдвиги на пару пикселей верстки.

Статистика: за неделю реальные тесты верстки UI через попиксельное сравнение скриншотов примерно 10% падений вызваны допустимой нестабильностью UI, а не реальными дефектами.

Оценка пользы (Расчеты примерные, чтобы показать смысл): время на анализ упавших тестов нестабильной верстки примерно составляет 10 минут за прогон, а обновления версий для тестирования может быть несколько в день. Для примерного рассчета возьмем 2 версии UI в день, то есть 2 х 10 = 20 минут в день. Или в месяц 21 х 20 / 60 = 7 часов в месяц. Дальше можно умножать на количество систем в компании, допустим 20 и 20 х 7 = 140 часов в месяц.

Решения на рынке есть, но они платные и обучены без контекста, ограничений и особенностей каждого проекта. 

## Цель
Создать решение на базе нейросети или модели, которая будет классифицировать 10% нестабильных случаев сравнения скриншотов верстки UI на наличие дефекта (True) и допустимую нестабильность UI (False). При этом нужно минимизировать долю ложных срабатываний (False Positive Rate) при сохранении высокой полноты (Recall) обнаружения реальных дефектов.

## Ограничения
При создании data set отсутсвует возможность получить реальные дефекты верстки UI, поэтому дефекты будем создавать синтетические. Так как их приходится делать руками, то их не много. Следовательно, виды дефектов тоже ограничены. Исследование направлено на оценку принципиальной возможности классификации, а для реального проекта оценки обобщающей способности потребуется валидация на реальных багах.

На реальных проектах проверки верстки UI через попиксельное сравнение скриншотов используется изолированная среда, например образ docker selenoid, чтобы исключить различия в размерах мониторов, разрешения, версий браузера и т.п. Поэтому и мы не будем рассматривать их изменение, а сосредоточимся на классификации дефект или нестабильность.

В реальных проектах динамические элементы можно закрасить перед сравнением, а баннеры закрыть. Но именно часть из нестабильных тестов падает из-за неожиданного баннера, лоадера и т.п, что не является дефектом.

Часто на реальных проектах нестабильность сравнения скриншотов решают за счет уменьшения процента сравнения, но тогда возможно упустить баги. Мы считаем, что перед моделью будет 100 % попиксельное сравнение скриншотов и если они совпадают, то использование модели или нейросети не нужно.

## Данные
Данные представляют из себя пару скриншотов верстки UI сайта netology:
- expected: ожидаемое состояние верстки UI;
- actual: текущее состояние верстки UI.

Data set разделен на два типа:
- bug: пара скриншотов expected и actual, в которых находится дефект верстки UI;
- not bug: пара скриншотов expected и actual, в которых есть разница, но дефекта нет.

Типы содержат в себе несколько видов дефектов и изменений:
- bug:
  - исчезновение элементов 100 пар скриншотов;
  - изменение цвета 50 пар скриншотов;
  - сдвиг элемента 50 пар скриншотов;
- not bug:
  - динамические элементы снятые в разное время 100 пар скриншотов;
  - неожидаемое появление баннеров 100 пар скриншотов;
  - небольшие сдвиги всей верстки, которые обусловлены искажениями перемотки страницы, появляются случайно в not bug и идеально воспроизводят реальную проблему нестабильности сравнения скриншотов.

### Итого: 200 скриншотов с дефектами и 200 скриншотов с разницей между скриншотами, но не являющиеся багами.